In [0]:
--Detailed Transaction View
CREATE OR REPLACE VIEW retail_catalog.retail_gold.vw_detailed_transactions AS
SELECT
    s.order_id,
    s.order_date,
    s.ship_date,
    s.ship_mode,

    c.customer_id,
    c.customer_name,
    c.segment,
    c.age,
    c.age_category,
    c.country,
    c.region,

    p.product_id,
    p.product_name,
    p.category,

    s.sales,
    s.quantity,
    s.discount,
    s.discount_amount,
    s.profit,
    s.profit_margin,
    s.order_size_category,
    s.shipping_days,
    s.profitability_category,

    s.order_year,
    s.order_month,
    s.order_quarter,
    s.order_week,
    s.day_of_week,
    s.is_weekend,

    s.data_arrival_timestamp

FROM delta.`/Volumes/retail_catalog/retail_gold/gold_sales_fact_volume` s
LEFT JOIN delta.`/Volumes/retail_catalog/retail_gold/gold_customer_dim_volume` c
    ON s.customer_id = c.customer_id
LEFT JOIN delta.`/Volumes/retail_catalog/retail_gold/gold_product_dim_volume` p
    ON s.product_id = p.product_id
WHERE s.is_deleted = false
  AND c.is_active = true
  AND p.is_active = true;

In [0]:
--sales by customer
CREATE OR REPLACE VIEW retail_catalog.retail_gold.vw_sales_by_customer AS
SELECT
    c.customer_id,
    c.customer_name,
    c.segment,
    c.region,
    COUNT(DISTINCT s.order_id) AS total_orders,
    SUM(s.sales) AS total_sales,
    SUM(s.profit) AS total_profit,
    SUM(s.quantity) AS total_quantity,
    AVG(s.profit_margin) AS avg_profit_margin,
    SUM(s.discount_amount) AS total_discount_given
FROM delta.`/Volumes/retail_catalog/retail_gold/gold_sales_fact_volume` s
JOIN delta.`/Volumes/retail_catalog/retail_gold/gold_customer_dim_volume` c
    ON s.customer_id = c.customer_id
WHERE s.is_deleted = false
  AND c.is_active = true
GROUP BY
    c.customer_id,
    c.customer_name,
    c.segment,
    c.region;

In [0]:
--product performance view
CREATE OR REPLACE VIEW retail_catalog.retail_gold.vw_product_performance AS
SELECT
    p.product_id,
    p.product_name,
    p.category,
    COUNT(DISTINCT s.order_id) AS total_orders,
    SUM(s.sales) AS total_sales,
    SUM(s.profit) AS total_profit,
    SUM(s.quantity) AS total_quantity,
    AVG(s.profit_margin) AS avg_profit_margin,
    SUM(s.discount_amount) AS total_discount,
    CASE 
        WHEN AVG(s.profit_margin) > 0.3 THEN 'High Margin'
        WHEN AVG(s.profit_margin) BETWEEN 0.15 AND 0.3 THEN 'Medium Margin'
        ELSE 'Low Margin'
    END AS margin_category
FROM delta.`/Volumes/retail_catalog/retail_gold/gold_sales_fact_volume` s
JOIN delta.`/Volumes/retail_catalog/retail_gold/gold_product_dim_volume` p
    ON s.product_id = p.product_id
WHERE s.is_deleted = false
  AND p.is_active = true
GROUP BY
    p.product_id,
    p.product_name,
    p.category;

In [0]:
--Monthly Sales Trend (Time Intelligence View)
CREATE OR REPLACE VIEW retail_catalog.retail_gold.vw_monthly_sales_trend AS
SELECT
    s.order_year,
    s.order_month,
    s.order_quarter,

    SUM(s.sales) AS total_sales,
    SUM(s.profit) AS total_profit,
    SUM(s.quantity) AS total_quantity,
    COUNT(DISTINCT s.order_id) AS total_orders,

    ROUND(SUM(s.profit) / SUM(s.sales), 4) AS overall_profit_margin

FROM delta.`/Volumes/retail_catalog/retail_gold/gold_sales_fact_volume` s
WHERE s.is_deleted = false
GROUP BY
    s.order_year,
    s.order_month,
    s.order_quarter
ORDER BY
    s.order_year,
    s.order_month;

In [0]:
--360° Sales Analytics View
CREATE OR REPLACE VIEW retail_catalog.retail_gold.vw_sales_360_analytics AS
WITH base AS (
    SELECT
        s.order_id,
        s.order_date,
        s.order_year,
        s.order_month,
        s.order_quarter,
        s.order_week,
        s.day_of_week,
        s.is_weekend,

        s.sales,
        s.quantity,
        s.discount,
        s.discount_amount,
        s.profit,
        s.profit_margin,
        s.shipping_days,

        c.customer_id,
        c.customer_name,
        c.segment,
        c.region,
        c.country,
        c.age,
        c.age_category,

        p.product_id,
        p.product_name,
        p.category

    FROM delta.`/Volumes/retail_catalog/retail_gold/gold_sales_fact_volume` s
    LEFT JOIN delta.`/Volumes/retail_catalog/retail_gold/gold_customer_dim_volume` c
        ON s.customer_id = c.customer_id
    LEFT JOIN delta.`/Volumes/retail_catalog/retail_gold/gold_product_dim_volume` p
        ON s.product_id = p.product_id
    WHERE s.is_deleted = false
      AND c.is_active = true
      AND p.is_active = true
),

aggregated AS (
    SELECT
        order_year,
        order_month,
        order_quarter,
        region,
        segment,
        category,

        COUNT(DISTINCT order_id) AS total_orders,
        SUM(sales) AS total_sales,
        SUM(quantity) AS total_quantity,
        SUM(profit) AS total_profit,
        SUM(discount_amount) AS total_discount,
        AVG(profit_margin) AS avg_profit_margin,
        AVG(shipping_days) AS avg_shipping_days,

        SUM(CASE WHEN is_weekend = true THEN sales ELSE 0 END) AS weekend_sales,
        SUM(CASE WHEN is_weekend = false THEN sales ELSE 0 END) AS weekday_sales

    FROM base
    GROUP BY
        order_year,
        order_month,
        order_quarter,
        region,
        segment,
        category
),

time_intelligence AS (
    SELECT
        *,
        
        -- YTD Sales
        SUM(total_sales) OVER (
            PARTITION BY order_year, region, segment, category
            ORDER BY order_month
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS ytd_sales,

        -- MTD Growth (MoM %)
        LAG(total_sales) OVER (
            PARTITION BY region, segment, category
            ORDER BY order_year, order_month
        ) AS previous_month_sales

    FROM aggregated
)

SELECT
    order_year,
    order_month,
    order_quarter,
    region,
    segment,
    category,

    total_orders,
    total_sales,
    total_quantity,
    total_profit,
    total_discount,
    avg_profit_margin,
    avg_shipping_days,

    weekend_sales,
    weekday_sales,

    ytd_sales,

    ROUND(
        (total_sales - previous_month_sales) / previous_month_sales,
        4
    ) AS mom_growth_percentage,

    -- Profitability Segmentation
    CASE 
        WHEN avg_profit_margin > 0.30 THEN 'High Margin'
        WHEN avg_profit_margin BETWEEN 0.15 AND 0.30 THEN 'Medium Margin'
        ELSE 'Low Margin'
    END AS profitability_segment,

    -- Sales Performance Segmentation
    CASE 
        WHEN total_sales > 100000 THEN 'Top Performer'
        WHEN total_sales BETWEEN 50000 AND 100000 THEN 'Mid Performer'
        ELSE 'Low Performer'
    END AS sales_segment

FROM time_intelligence
ORDER BY order_year, order_month;


### using all views below

In [0]:
select * from retail_catalog.retail_gold.vw_detailed_transactions;

In [0]:
select * from retail_catalog.retail_gold.vw_monthly_sales_trend;

In [0]:
select * from retail_catalog.retail_gold.vw_product_performance;

In [0]:
select * from retail_catalog.retail_gold.vw_sales_360_analytics;

In [0]:
select * from retail_catalog.retail_gold.vw_sales_by_customer;